# Sentiment Prediction using the Trained Simple RNN Model

## Import Required Libraries

```python
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import load_model
```

### Explanation

These libraries are used to load the trained model, preprocess user input, and perform sentiment prediction.

- **NumPy** → Numerical operations.
- **TensorFlow** → Deep Learning framework.
- **IMDb Dataset** → Provides the word dictionary.
- **sequence** → Used for padding input reviews.
- **load_model()** → Loads the trained `.h5` model.

---

# Load the Trained Model

```python
model = load_model('simple_rnn_imdb.h5')
```

### Explanation

Loads the trained Simple RNN model from the saved `.h5` file.

The loaded model already contains:

- Model architecture
- Learned weights
- Optimizer information

This means we **do not need to train the model again**.

---

# Display Model Summary

```python
model.summary()
```

### Explanation

Displays information about the trained model.

It shows:

- Layers
- Output shapes
- Number of trainable parameters

Useful for verifying that the correct model has been loaded.

---

# Load the IMDb Word Dictionary

```python
word_index = imdb.get_word_index()
```

### Explanation

Loads the IMDb vocabulary.

This dictionary maps

```text
Word

↓

Integer ID
```

Example

```text
movie → 17

good → 49

bad → 76
```

The same dictionary was used while preparing the training dataset.

---

# Create Reverse Dictionary

```python
reverse_word_index = {
    value: key
    for key, value in word_index.items()
}
```

### Explanation

Creates the opposite mapping.

Original

```text
Word → Integer
```

Reverse

```text
Integer → Word
```

This is useful for converting encoded reviews back into readable text.

---

# Decode Review Function

```python
def decode_review(encoded_review):
    return ' '.join(
        [reverse_word_index.get(i - 3, '?')
         for i in encoded_review]
    )
```

### Explanation

The IMDb dataset stores reviews as integer IDs.

Example

```text
[1, 14, 20, 530]
```

This function converts those integers back into words.

Steps:

1. Read one integer at a time.
2. Convert it into the corresponding word.
3. Join all words into a sentence.

The review becomes readable.

---

# Why `i - 3`?

IMDb reserves the first few indices for special tokens.

Examples include:

- Padding
- Start of Review
- Unknown Word

Therefore,

```python
i - 3
```

adjusts the index before searching the dictionary.

---

# Preprocess User Input

```python
def preprocess_text(text):
```

This function prepares a user-entered review before sending it to the model.

---

## Convert to Lowercase

```python
words = text.lower().split()
```

### Explanation

Example

Before

```text
This Movie Was GREAT
```

After

```text
this movie was great
```

Then `split()` breaks the sentence into words.

```text
['this', 'movie', 'was', 'great']
```

---

## Convert Words into Integer IDs

```python
encoded_review = [
    word_index.get(word, 2) + 3
    for word in words
]
```

### Explanation

Every word is converted into its IMDb integer ID.

Example

```text
this

↓

13

movie

↓

17
```

If a word does not exist in the dictionary,

```python
word_index.get(word, 2)
```

returns

```text
2
```

which represents an **Unknown Word (OOV)**.

Again,

```python
+3
```

adjusts the indices because of IMDb's reserved tokens.

---

## Apply Padding

```python
padded_review = sequence.pad_sequences(
    [encoded_review],
    maxlen=500
)
```

### Explanation

The trained model expects every review to contain exactly **500 words**.

If the review is shorter,

zeros are added.

If the review is longer,

extra words are removed.

Output shape

```text
(1, 500)
```

This is exactly the input shape expected by the Simple RNN.

---

## Return the Processed Review

```python
return padded_review
```

The processed review is now ready for prediction.

---

# View Model Weights

```python
model.get_weights()
```

### Explanation

Returns all the learned weights of the trained neural network.

These include:

- Embedding weights
- RNN weights
- Dense layer weights

Usually used for debugging or inspection.

---

# Prediction Function

```python
def predict_sentiment(review):
```

This function predicts whether a movie review is Positive or Negative.

---

## Preprocess Input

```python
preprocessed_input = preprocess_text(review)
```

Converts raw text into padded integer sequences.

---

## Predict

```python
prediction = model.predict(preprocessed_input)
```

The review passes through

```text
Embedding

↓

Simple RNN

↓

Dense Layer

↓

Sigmoid
```

The model outputs a probability.

Example

```text
0.92
```

---

## Convert Probability into Sentiment

```python
sentiment = (
    "Positive"
    if prediction[0][0] > 0.5
    else "Negative"
)
```

### Explanation

Decision Rule

```text
Prediction > 0.5

↓

Positive
```

```text
Prediction ≤ 0.5

↓

Negative
```

---

## Return Result

```python
return sentiment, prediction[0][0]
```

Returns

- Predicted Sentiment
- Prediction Probability

---

# Predict a New Review

```python
example_review = "This movie was fantastic! The acting was great and the plot was thrilling."
```

This is a completely new review entered by the user.

---

# Make Prediction

```python
sentiment, score = predict_sentiment(example_review)
```

The complete prediction pipeline is executed.

```text
User Review
      ↓
Lowercase
      ↓
Split into Words
      ↓
Word IDs
      ↓
Padding
      ↓
Embedding Layer
      ↓
Simple RNN
      ↓
Sigmoid
      ↓
Probability
      ↓
Positive / Negative
```

---

# Display the Result

```python
print(f"Review: {example_review}")
print(f"Sentiment: {sentiment}")
print(f"Prediction Score: {score}")
```

### Example Output

```text
Review:
This movie was fantastic! The acting was great and the plot was thrilling.

Sentiment:
Positive

Prediction Score:
0.96
```

A score close to **1** indicates high confidence for a positive review.

A score close to **0** indicates high confidence for a negative review.

---

# Complete Workflow

```text
User Review
      ↓
Lowercase & Split
      ↓
Convert Words to Integer IDs
      ↓
Padding (500 words)
      ↓
Load Trained Simple RNN
      ↓
Embedding Layer
      ↓
Simple RNN
      ↓
Dense + Sigmoid
      ↓
Prediction Probability
      ↓
Positive / Negative Sentiment
```

---

# Key Points

- `load_model()` loads the trained `.h5` model without retraining.
- `word_index` converts words into integer IDs.
- `reverse_word_index` converts integer IDs back into readable words.
- `preprocess_text()` prepares user input exactly like the training data.
- `pad_sequences()` ensures every review has a fixed length of 500 words.
- `model.predict()` returns a probability between **0 and 1**.
- If the probability is greater than **0.5**, the review is classified as **Positive**; otherwise, it is classified as **Negative**.
- The prediction pipeline must use the **same preprocessing steps** as the training pipeline to produce reliable results.

In [1]:
# Step 1: Import Libraries and Load the Model
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import load_model

In [2]:

# Load the IMDB dataset word index
word_index = imdb.get_word_index()
reverse_word_index = {value: key for key, value in word_index.items()}

In [3]:
# Load the pre-trained model with ReLU activation
model = load_model('simple_rnn_imdb.h5')
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (32, 500, 128)         │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (32, 128)              │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (32, 1)                │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,027 (5.01 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [4]:
model.get_weights()

[array([[-0.41130978,  0.2362482 ,  0.12507695, ...,  0.06026619,
         -0.3272072 ,  0.39754653],
        [-0.04515494,  0.09930022,  0.06200958, ..., -0.04230523,
         -0.01065478,  0.03936344],
        [-0.08585467,  0.01542323,  0.03298138, ..., -0.05501086,
         -0.12996776,  0.08282567],
        ...,
        [ 0.10789968,  0.07200178,  0.01574387, ...,  0.0742645 ,
         -0.02129543,  0.0319366 ],
        [-0.04970637, -0.00642039, -0.08720731, ..., -0.02178101,
         -0.00604324,  0.06039384],
        [ 0.08472186,  0.01444197,  0.00456899, ...,  0.06831315,
          0.02053973,  0.00049074]], shape=(10000, 128), dtype=float32),
 array([[ 0.00653101,  0.07552227,  0.12249252, ...,  0.01929918,
          0.11938007, -0.13208655],
        [-0.00617964, -0.03738829, -0.07178559, ...,  0.01071848,
          0.05640419, -0.02411353],
        [-0.16688477,  0.004966  , -0.02412383, ..., -0.02781708,
         -0.07067017, -0.14147584],
        ...,
        [ 0.0254812

In [5]:
# Step 2: Helper Functions
# Function to decode reviews
def decode_review(encoded_review):
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_review])

# Function to preprocess user input
def preprocess_text(text):
    words = text.lower().split()
    encoded_review = [word_index.get(word, 2) + 3 for word in words]
    padded_review = sequence.pad_sequences([encoded_review], maxlen=500)
    return padded_review

In [6]:
### Prediction  function

def predict_sentiment(review):
    preprocessed_input=preprocess_text(review)

    prediction=model.predict(preprocessed_input)

    sentiment = 'Positive' if prediction[0][0] > 0.5 else 'Negative'
    
    return sentiment, prediction[0][0]



In [17]:
# Step 4: User Input and Prediction
# Example review for prediction
example_review = "One of the worst movies I have ever seen. It was boring, slow and completely disappointing."

sentiment,score=predict_sentiment(example_review)

print(f'Review: {example_review}')
print(f'Sentiment: {sentiment}')
print(f'Prediction Score: {score}')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
Review: One of the worst movies I have ever seen. It was boring, slow and completely disappointing.
Sentiment: Positive
Prediction Score: 0.6256012320518494
